In [0]:
# Configuración general del laboratorio
 
from datetime import datetime, UTC
from pyspark.sql.functions import col
import re
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
 
CATALOG = "workspace"
SCHEMA = "bigdata_proyecto"
 
BRONZE_TABLE_Ventas = f"{CATALOG}.{SCHEMA}.BRONZE_Ventas_Proyecto"
BRONZE_TABLE_Clientes = f"{CATALOG}.{SCHEMA}.BRONZE_Clientes_Proyecto"
BRONZE_TABLE_Provedores = f"{CATALOG}.{SCHEMA}.BRONZE_Provedores_Proyecto"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.BRONZE_GENERAL"

SILVER_TABLE = f"{CATALOG}.{SCHEMA}.SILVER_Ventas_Proyecto"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.GOLD_Ventas_Proyecto"
 
# Ruta del CSV entregado a estudiantes
# Ajusta esta ruta según donde hayas dejado el archivo en Databricks

CSV_Ventas = "Ventas.csv"
CSV_Clientes = "Clientes.csv"
CSV_Provedores = "Provedores.csv"

CSV_INPUT_PATH = "/Volumes/workspace/bigdata_proyecto/proyecto_bigdata/"
 
RUN_TS = datetime.now(UTC)
RUN_DATE = RUN_TS.strftime("%Y-%m-%d")
 
print("Bronze table ventas:", BRONZE_TABLE_Ventas)
print("Bronze table clientes:", BRONZE_TABLE_Clientes)
print("Bronze table provedores:", BRONZE_TABLE_Provedores)
print("Bronze table Final:", BRONZE_TABLE)
print("---------------------------------------------------")
print("Run date:", RUN_DATE)
print("Silver table:", SILVER_TABLE)
print("Gold table:", GOLD_TABLE)
print("---------------------------------------------------")
print("CSV input path:", CSV_INPUT_PATH)
print("CSV Ventas input path:", CSV_INPUT_PATH +        CSV_Ventas)
print("CSV Clientes input path:", CSV_INPUT_PATH +      CSV_Clientes)
print("CSV Provedores input path:", CSV_INPUT_PATH +    CSV_Provedores)
print("BRONCE FINAL input path:", CSV_INPUT_PATH +    BRONZE_TABLE)

Bronze table ventas: workspace.bigdata_proyecto.BRONZE_Ventas_Proyecto
Bronze table clientes: workspace.bigdata_proyecto.BRONZE_Clientes_Proyecto
Bronze table provedores: workspace.bigdata_proyecto.BRONZE_Provedores_Proyecto
Bronze table Final: workspace.bigdata_proyecto.BRONZE_GENERAL
---------------------------------------------------
Run date: 2026-05-10
Silver table: workspace.bigdata_proyecto.SILVER_Ventas_Proyecto
Gold table: workspace.bigdata_proyecto.GOLD_Ventas_Proyecto
---------------------------------------------------
CSV input path: /Volumes/workspace/bigdata_proyecto/proyecto_bigdata/
CSV Ventas input path: /Volumes/workspace/bigdata_proyecto/proyecto_bigdata/Ventas.csv
CSV Clientes input path: /Volumes/workspace/bigdata_proyecto/proyecto_bigdata/Clientes.csv
CSV Provedores input path: /Volumes/workspace/bigdata_proyecto/proyecto_bigdata/Provedores.csv
BRONCE FINAL input path: /Volumes/workspace/bigdata_proyecto/proyecto_bigdata/workspace.bigdata_proyecto.BRONZE_GENERAL


In [0]:
df_raw_CSV_Ventas = (spark.read.option("header", True).option("inferSchema", True).option("delimiter", ";").csv(CSV_INPUT_PATH + CSV_Ventas))
df_raw_CSV_Clientes = (spark.read.option("header", True).option("inferSchema", True).option("delimiter", ";").csv(CSV_INPUT_PATH + CSV_Clientes))
df_raw_CSV_Provedores = (spark.read.option("header", True).option("inferSchema", True).option("delimiter", ";").csv(CSV_INPUT_PATH + CSV_Provedores )) 
display(df_raw_CSV_Ventas.limit(5))
display(df_raw_CSV_Clientes.limit(5))
display(df_raw_CSV_Provedores.limit(5))

ID Cliente,id provedor,Zona,País,Tipo de producto,Canal de venta,Prioridad,Fecha pedido,ID Pedido,Fecha envío,Unidades,Precio Unitario,Coste unitario,Importe venta total,Importe Coste total
C2421,P0001,Europa,United Kingdom,Snacks,Offline,Crítica,2011-10-12,20111012-8455,2011-11-30,"84173,00","159,45","101,82","13421056,58","8570898,89"
C1908,P0002,Europa,Malta,Cárnicos,Online,Alta,2011-01-26,20110126-3902,2011-01-28,"19314,00","440,88","381,10","8515060,72","7360585,68"
C7652,P0003,Australia y Oceanía,Marshall Islands,Cereales,Online,Crítica,2011-11-09,20111109-1249,2011-11-21,"2180,00","214,96","122,38","468605,17","266788,29"
C2326,P0004,África,Iran,Frutas,Offline,Baja,2012-08-21,20120821-8949,2012-10-02,"20756,00","9,81","7,27","203529,81","150956,73"
C5305,P0005,Centroamérica y Caribe,Guatemala,Alimento infantil,Offline,Media,2013-09-30,20130930-4675,2013-11-12,"73533,00","260,64","162,77","19165705,83","11968806,11"


ID,Nombre completo,Fecha de nacimiento,Direcci�n,Localidad y C�digo postal,Tel�fono,Correo electr�nico,Fecha de alta,Grupo de clientes
C2421,Leandra Anna Malo Alba,1984-12-08,7943 S. Fifth Street,"Bergenfield, NJ 07621",(598) 451-5865,uraeus@mac.com,19/01/2012 14:32,A
C1908,Severo Granados Iglesia,1986-08-12,77 Lyme Street,"Hermitage, TN 37076",(869) 771-1487,bhima@me.com,22/03/2005 15:42,E
C7652,Lucho Andreu Amat,1990-04-16,9448 Fairfield St.,"Aberdeen, SD 57401",(246) 245-7306,psichel@sbcglobal.net,15/09/2007 3:01,E
C2326,Mat�as Mauricio Castillo Barrera,1996-12-02,8143 College St.,"Trussville, AL 35173",(707) 933-2513,tbeck@optonline.net,7/12/2011 15:22,E
C5305,Mauricio Guijarro Castell�,1984-05-14,9893 W. Vale Ave.,"Billings, MT 59101",(612) 325-0216,eegsa@yahoo.ca,28/06/2008 6:58,D


ID,Proveedor,Contacto comercial,Email,Teléfono,Saldo pendiente,Fecha de última compra,_c7
P0001,So Factive,Lorenzo Cantón Galan,fluffy@verizon.net,(394) 406-8708,"63,00",2018-09-04,null
P0002,Kontroller,Adelina Valls Canet,fraterk@me.com,(923) 207-3871,"4250,00",2012-06-08,null
P0003,Finance Api,Eulalia del Galindo,animats@mac.com,(904) 363-2261,"2412,00",2017-08-06,null
P0004,Biomotivate,Tania Catalán Galván,tbeck@icloud.com,(923) 671-4117,"3348,00",2017-05-22,null
P0005,Deltavita,Abraham Girón-Soler,mallanmba@yahoo.ca,(841) 798-8943,"570,00",2022-01-17,null


In [0]:
df_raw_CSV_Ventas.printSchema()
print("Total de registros:", df_raw_CSV_Ventas.count())
print("Total de pedidos:", df_raw_CSV_Ventas.select("ID Pedido").distinct().count())
 

root
 |-- ID Cliente: string (nullable = true)
 |-- id provedor: string (nullable = true)
 |-- Zona: string (nullable = true)
 |-- País: string (nullable = true)
 |-- Tipo de producto: string (nullable = true)
 |-- Canal de venta: string (nullable = true)
 |-- Prioridad: string (nullable = true)
 |-- Fecha pedido: date (nullable = true)
 |-- ID Pedido: string (nullable = true)
 |-- Fecha envío: date (nullable = true)
 |-- Unidades: string (nullable = true)
 |-- Precio Unitario: string (nullable = true)
 |-- Coste unitario: string (nullable = true)
 |-- Importe venta total: string (nullable = true)
 |-- Importe Coste total: string (nullable = true)

Total de registros: 12000
Total de pedidos: 12000


In [0]:
df_raw_CSV_Clientes.printSchema()
print("Total de registros:", df_raw_CSV_Clientes.count())
print("Total clientes:", df_raw_CSV_Clientes.select("ID").distinct().count())

root
 |-- ID: string (nullable = true)
 |-- Nombre completo: string (nullable = true)
 |-- Fecha de nacimiento: date (nullable = true)
 |-- Direcci�n: string (nullable = true)
 |-- Localidad y C�digo postal: string (nullable = true)
 |-- Tel�fono: string (nullable = true)
 |-- Correo electr�nico: string (nullable = true)
 |-- Fecha de alta: string (nullable = true)
 |-- Grupo de clientes: string (nullable = true)

Total de registros: 999
Total clientes: 999


In [0]:
df_raw_CSV_Provedores.printSchema()
print("Total de registros:", df_raw_CSV_Provedores.count())
print("Total provedores:", df_raw_CSV_Provedores.select("Proveedor").distinct().count())

root
 |-- ID: string (nullable = true)
 |-- Proveedor: string (nullable = true)
 |-- Contacto comercial: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Teléfono: string (nullable = true)
 |-- Saldo pendiente: string (nullable = true)
 |-- Fecha de última compra: date (nullable = true)
 |-- _c7: string (nullable = true)

Total de registros: 999
Total provedores: 999


%md

# Construcción de la Capa Bronze (`Build_Bronze_Layer`)

## Descripción Funcional
Este componente transforma el DataFrame crudo incorporando metadatos de ingestión y persiste el resultado en formato Delta Lake como tabla Bronze. Se añaden campos de trazabilidad para identificar el origen y el momento de carga de los datos.

## Objetivo dentro del Pipeline
Materializar la capa Bronze dentro de la arquitectura Medallion, garantizando almacenamiento persistente de datos en estado casi crudo, enriquecidos con información mínima de auditoría.

## Entradas
- **DataFrame origen:** `df_raw_csv`
- **Configuración:**
  - Tabla destino: `BRONZE_TABLE`
  - Formato: Delta Lake

## Procesamiento
- Eliminación previa de la tabla Bronze (si existe):
  - `DROP TABLE IF EXISTS`
- Enriquecimiento del DataFrame:
  - `ingestion_timestamp`: timestamp actual de carga
  - `record_source`: identificador del origen (`csv_docente`)
- Escritura del DataFrame:
  - Formato: `delta`
  - Modo: `overwrite`
  - Persistencia como tabla gestionada

## Salidas
- **Tabla Delta:** `BRONZE_TABLE`
- Datos en estado crudo con metadatos de ingestión

## Capa del Pipeline
**Bronze**

## Consideraciones Técnicas
- **Overwrite completo:** Elimina versiones anteriores (no incremental)
- **Trazabilidad:** Se incorporan columnas clave para auditoría
- **Formato Delta:**
  - Soporte para versionado
  - ACID transactions
  - Optimización de consultas
- **Escalabilidad:** Aprovecha procesamiento distribuido de Spark
- **Riesgo potencial:** Pérdida de histórico si no se maneja versionado (Time Travel)



In [0]:
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE_Ventas}")
 
df_bronze_Ventas = (
    df_raw_CSV_Ventas
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_source", F.lit("csv_ventas"))
)
 
# Limpiar nombres de columnas
nuevas_columnas = [
    re.sub(r'[ ,;{}()\n\t=]+', '_', c)
    for c in df_bronze_Ventas.columns
]
# Aplicar nuevos nombres
df_bronze_Ventas = df_bronze_Ventas.toDF(*nuevas_columnas)

(
    df_bronze_Ventas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE_Ventas)
)
 
print(f"Tabla Bronze creada: {BRONZE_TABLE_Ventas}")

Tabla Bronze creada: workspace.bigdata_proyecto.BRONZE_Ventas_Proyecto


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE_Clientes}")
 
df_bronze_Clientes = (
    df_raw_CSV_Clientes
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_source", F.lit("csv_clientes"))
)

# Limpiar nombres de columnas
nuevas_columnas = [
    re.sub(r'[ ,;{}()\n\t=]+', '_', c)
    for c in df_bronze_Clientes.columns
]
# Aplicar nuevos nombres
df_bronze_Clientes = df_bronze_Clientes.toDF(*nuevas_columnas)

(
    df_bronze_Clientes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE_Clientes)
)
 
print(f"Tabla Bronze creada: {BRONZE_TABLE_Clientes}")

Tabla Bronze creada: workspace.bigdata_proyecto.BRONZE_Clientes_Proyecto


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE_Provedores}")
 
df_bronze_Provedores = (
    df_raw_CSV_Provedores
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_source", F.lit("csv_provedores"))
)

# Limpiar nombres de columnas
nuevas_columnas = [
    re.sub(r'[ ,;{}()\n\t=]+', '_', c)
    for c in df_bronze_Provedores.columns
]
# Aplicar nuevos nombres
df_bronze_Provedores = df_bronze_Provedores.toDF(*nuevas_columnas)

(
    df_bronze_Provedores.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE_Provedores)
)
 
print(f"Tabla Bronze creada: {BRONZE_TABLE_Provedores}")

Tabla Bronze creada: workspace.bigdata_proyecto.BRONZE_Provedores_Proyecto


In [0]:
display(df_bronze_Ventas.limit(2))
display(df_bronze_Clientes.limit(2))
display(df_bronze_Provedores.limit(2))

ID_Cliente,id_provedor,Zona,País,Tipo_de_producto,Canal_de_venta,Prioridad,Fecha_pedido,ID_Pedido,Fecha_envío,Unidades,Precio_Unitario,Coste_unitario,Importe_venta_total,Importe_Coste_total,ingestion_timestamp,record_source
C2421,P0001,Europa,United Kingdom,Snacks,Offline,Crítica,2011-10-12,20111012-8455,2011-11-30,"84173,00","159,45","101,82","13421056,58","8570898,89",2026-05-10T03:50:00.858Z,csv_ventas
C1908,P0002,Europa,Malta,Cárnicos,Online,Alta,2011-01-26,20110126-3902,2011-01-28,"19314,00","440,88","381,10","8515060,72","7360585,68",2026-05-10T03:50:00.858Z,csv_ventas


ID,Nombre_completo,Fecha_de_nacimiento,Direcci�n,Localidad_y_C�digo_postal,Tel�fono,Correo_electr�nico,Fecha_de_alta,Grupo_de_clientes,ingestion_timestamp,record_source
C2421,Leandra Anna Malo Alba,1984-12-08,7943 S. Fifth Street,"Bergenfield, NJ 07621",(598) 451-5865,uraeus@mac.com,19/01/2012 14:32,A,2026-05-10T03:50:01.335Z,csv_clientes
C1908,Severo Granados Iglesia,1986-08-12,77 Lyme Street,"Hermitage, TN 37076",(869) 771-1487,bhima@me.com,22/03/2005 15:42,E,2026-05-10T03:50:01.335Z,csv_clientes


ID,Proveedor,Contacto_comercial,Email,Teléfono,Saldo_pendiente,Fecha_de_última_compra,_c7,ingestion_timestamp,record_source
P0001,So Factive,Lorenzo Cantón Galan,fluffy@verizon.net,(394) 406-8708,"63,00",2018-09-04,null,2026-05-10T03:50:01.840Z,csv_provedores
P0002,Kontroller,Adelina Valls Canet,fraterk@me.com,(923) 207-3871,"4250,00",2012-06-08,null,2026-05-10T03:50:01.840Z,csv_provedores


In [0]:


display(spark.table(BRONZE_TABLE_Ventas).limit(2).orderBy("ID_Cliente", "Fecha_envío"))
display(spark.table(BRONZE_TABLE_Clientes).limit(2).orderBy("ID", "Fecha_de_alta"))
display(spark.table(BRONZE_TABLE_Provedores).limit(2).orderBy("ID", "ingestion_timestamp"))

ID_Cliente,id_provedor,Zona,País,Tipo_de_producto,Canal_de_venta,Prioridad,Fecha_pedido,ID_Pedido,Fecha_envío,Unidades,Precio_Unitario,Coste_unitario,Importe_venta_total,Importe_Coste_total,ingestion_timestamp,record_source
C1908,P0002,Europa,Malta,Cárnicos,Online,Alta,2011-01-26,20110126-3902,2011-01-28,"19314,00","440,88","381,10","8515060,72","7360585,68",2026-05-10T03:49:50.683Z,csv_ventas
C2421,P0001,Europa,United Kingdom,Snacks,Offline,Crítica,2011-10-12,20111012-8455,2011-11-30,"84173,00","159,45","101,82","13421056,58","8570898,89",2026-05-10T03:49:50.683Z,csv_ventas


ID,Nombre_completo,Fecha_de_nacimiento,Direcci�n,Localidad_y_C�digo_postal,Tel�fono,Correo_electr�nico,Fecha_de_alta,Grupo_de_clientes,ingestion_timestamp,record_source
C1908,Severo Granados Iglesia,1986-08-12,77 Lyme Street,"Hermitage, TN 37076",(869) 771-1487,bhima@me.com,22/03/2005 15:42,E,2026-05-10T03:49:54.514Z,csv_clientes
C2421,Leandra Anna Malo Alba,1984-12-08,7943 S. Fifth Street,"Bergenfield, NJ 07621",(598) 451-5865,uraeus@mac.com,19/01/2012 14:32,A,2026-05-10T03:49:54.514Z,csv_clientes


ID,Proveedor,Contacto_comercial,Email,Teléfono,Saldo_pendiente,Fecha_de_última_compra,_c7,ingestion_timestamp,record_source
P0001,So Factive,Lorenzo Cantón Galan,fluffy@verizon.net,(394) 406-8708,"63,00",2018-09-04,null,2026-05-10T03:49:59.146Z,csv_provedores
P0002,Kontroller,Adelina Valls Canet,fraterk@me.com,(923) 207-3871,"4250,00",2012-06-08,null,2026-05-10T03:49:59.146Z,csv_provedores


## Observación de Data Engineering
Este componente implementa correctamente el patrón Bronze al preservar los datos en su forma original, añadiendo únicamente metadatos técnicos. En escenarios productivos, se recomienda evolucionar hacia cargas incrementales (append) y particionamiento por fecha de ingestión para mejorar performance y gobernanza.

Se realiza la carga de tres fuentes distintas (Ventas, Clientes y Proveedores) aplicando un inferSchema inicial para capturar los tipos de datos. Posteriormente, se realiza un Join Multidimensional para consolidar toda la información en una única tabla BRONZE_GENERAL. Se han añadido prefijos (CL_ y PR_) para evitar colisiones de nombres de columnas.

## Conclusion
El output del display(DF_BRONZE_CONSOLIDADO) revela una estructura rica en variables geográficas y temporales. Sin embargo, se detectan formatos de fecha heterogéneos y columnas con valores nulos (como _c7 en proveedores), lo que justifica técnicamente la necesidad de una etapa de limpieza profunda en la capa Silver.

In [0]:
# DataFrames origen
df_ventas = spark.table(BRONZE_TABLE_Ventas).alias("v")
df_clientes = spark.table(BRONZE_TABLE_Clientes).alias("c")
df_proveedores = spark.table(BRONZE_TABLE_Provedores).alias("p")

# Columnas de Ventas
cols_ventas = [col(f"v.{c}") for c in df_ventas.columns]

# Columnas de Clientes con prefijo CL_
cols_clientes = [
    col(f"c.{c}").alias(f"CL_{c}")
    for c in df_clientes.columns
]

# Columnas de Proveedores con prefijo PR_
cols_proveedores = [
    col(f"p.{c}").alias(f"PR_{c}")
    for c in df_proveedores.columns
]

# Modelo final
DF_BRONZE_CONSOLIDADO = (
    df_ventas

    .join(
        df_clientes,
        col("v.ID_Cliente") == col("c.ID"),
        "inner"
    )

    .join(
        df_proveedores,
        col("v.id_provedor") == col("p.ID"),
        "inner"
    )

    .select(
        *cols_ventas,
        *cols_clientes,
        *cols_proveedores
    )
)

display(DF_BRONZE_CONSOLIDADO.limit(5))

ID_Cliente,id_provedor,Zona,País,Tipo_de_producto,Canal_de_venta,Prioridad,Fecha_pedido,ID_Pedido,Fecha_envío,Unidades,Precio_Unitario,Coste_unitario,Importe_venta_total,Importe_Coste_total,ingestion_timestamp,record_source,CL_ID,CL_Nombre_completo,CL_Fecha_de_nacimiento,CL_Direcci�n,CL_Localidad_y_C�digo_postal,CL_Tel�fono,CL_Correo_electr�nico,CL_Fecha_de_alta,CL_Grupo_de_clientes,CL_ingestion_timestamp,CL_record_source,PR_ID,PR_Proveedor,PR_Contacto_comercial,PR_Email,PR_Teléfono,PR_Saldo_pendiente,PR_Fecha_de_última_compra,PR__c7,PR_ingestion_timestamp,PR_record_source
C2421,P0001,Europa,United Kingdom,Snacks,Offline,Crítica,2011-10-12,20111012-8455,2011-11-30,"84173,00","159,45","101,82","13421056,58","8570898,89",2026-05-10T03:49:50.683Z,csv_ventas,C2421,Leandra Anna Malo Alba,1984-12-08,7943 S. Fifth Street,"Bergenfield, NJ 07621",(598) 451-5865,uraeus@mac.com,19/01/2012 14:32,A,2026-05-10T03:49:54.514Z,csv_clientes,P0001,So Factive,Lorenzo Cantón Galan,fluffy@verizon.net,(394) 406-8708,"63,00",2018-09-04,null,2026-05-10T03:49:59.146Z,csv_provedores
C1908,P0002,Europa,Malta,Cárnicos,Online,Alta,2011-01-26,20110126-3902,2011-01-28,"19314,00","440,88","381,10","8515060,72","7360585,68",2026-05-10T03:49:50.683Z,csv_ventas,C1908,Severo Granados Iglesia,1986-08-12,77 Lyme Street,"Hermitage, TN 37076",(869) 771-1487,bhima@me.com,22/03/2005 15:42,E,2026-05-10T03:49:54.514Z,csv_clientes,P0002,Kontroller,Adelina Valls Canet,fraterk@me.com,(923) 207-3871,"4250,00",2012-06-08,null,2026-05-10T03:49:59.146Z,csv_provedores
C7652,P0003,Australia y Oceanía,Marshall Islands,Cereales,Online,Crítica,2011-11-09,20111109-1249,2011-11-21,"2180,00","214,96","122,38","468605,17","266788,29",2026-05-10T03:49:50.683Z,csv_ventas,C7652,Lucho Andreu Amat,1990-04-16,9448 Fairfield St.,"Aberdeen, SD 57401",(246) 245-7306,psichel@sbcglobal.net,15/09/2007 3:01,E,2026-05-10T03:49:54.514Z,csv_clientes,P0003,Finance Api,Eulalia del Galindo,animats@mac.com,(904) 363-2261,"2412,00",2017-08-06,null,2026-05-10T03:49:59.146Z,csv_provedores
C2326,P0004,África,Iran,Frutas,Offline,Baja,2012-08-21,20120821-8949,2012-10-02,"20756,00","9,81","7,27","203529,81","150956,73",2026-05-10T03:49:50.683Z,csv_ventas,C2326,Mat�as Mauricio Castillo Barrera,1996-12-02,8143 College St.,"Trussville, AL 35173",(707) 933-2513,tbeck@optonline.net,7/12/2011 15:22,E,2026-05-10T03:49:54.514Z,csv_clientes,P0004,Biomotivate,Tania Catalán Galván,tbeck@icloud.com,(923) 671-4117,"3348,00",2017-05-22,null,2026-05-10T03:49:59.146Z,csv_provedores
C5305,P0005,Centroamérica y Caribe,Guatemala,Alimento infantil,Offline,Media,2013-09-30,20130930-4675,2013-11-12,"73533,00","260,64","162,77","19165705,83","11968806,11",2026-05-10T03:49:50.683Z,csv_ventas,C5305,Mauricio Guijarro Castell�,1984-05-14,9893 W. Vale Ave.,"Billings, MT 59101",(612) 325-0216,eegsa@yahoo.ca,28/06/2008 6:58,D,2026-05-10T03:49:54.514Z,csv_clientes,P0005,Deltavita,Abraham Girón-Soler,mallanmba@yahoo.ca,(841) 798-8943,"570,00",2022-01-17,null,2026-05-10T03:49:59.146Z,csv_provedores


In [0]:
print(BRONZE_TABLE)

workspace.bigdata_proyecto.BRONZE_GENERAL


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")
 
df_bronze_CONSOLIDADO = (
    DF_BRONZE_CONSOLIDADO
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_source", F.lit("csv_general"))
)


(
    df_bronze_CONSOLIDADO.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)
 
print(f"Tabla Bronze creada: {BRONZE_TABLE}")

Tabla Bronze creada: workspace.bigdata_proyecto.BRONZE_GENERAL


In [0]:
display(spark.table(BRONZE_TABLE).orderBy("Fecha_pedido", "Fecha_envío").limit(5))

ID_Cliente,id_provedor,Zona,País,Tipo_de_producto,Canal_de_venta,Prioridad,Fecha_pedido,ID_Pedido,Fecha_envío,Unidades,Precio_Unitario,Coste_unitario,Importe_venta_total,Importe_Coste_total,ingestion_timestamp,record_source,CL_ID,CL_Nombre_completo,CL_Fecha_de_nacimiento,CL_Direcci�n,CL_Localidad_y_C�digo_postal,CL_Tel�fono,CL_Correo_electr�nico,CL_Fecha_de_alta,CL_Grupo_de_clientes,CL_ingestion_timestamp,CL_record_source,PR_ID,PR_Proveedor,PR_Contacto_comercial,PR_Email,PR_Teléfono,PR_Saldo_pendiente,PR_Fecha_de_última_compra,PR__c7,PR_ingestion_timestamp,PR_record_source
C4303,P0773,Centroamérica y Caribe,Trinidad and Tobago,Snacks,Online,Alta,2009-01-01,20090101-7221,2009-01-27,"63693,00","157,31","100,46","10019544,56","6398639,54",2026-05-10T03:50:10.843Z,csv_general,C4303,Demetrio de Larra�aga,1995-10-01,16 George Street,London,(516) 973-1052,mallanmba@comcast.net,21/01/2015 16:02,D,2026-05-10T03:49:54.514Z,csv_clientes,P0773,Nuovo Vr,Alfonso Vinicio Montoya Diaz,bogjobber@verizon.net,(745) 515-7670,"4982,00",2012-01-25,null,2026-05-10T03:49:59.146Z,csv_provedores
C4303,P0770,Centroamérica y Caribe,Trinidad and Tobago,Snacks,Online,Alta,2009-01-01,20090101-5425,2009-01-27,"77291,00","157,31","100,46","12158645,66","7764703,33",2026-05-10T03:50:10.843Z,csv_general,C4303,Demetrio de Larra�aga,1995-10-01,16 George Street,London,(516) 973-1052,mallanmba@comcast.net,21/01/2015 16:02,D,2026-05-10T03:49:54.514Z,csv_clientes,P0770,Vision Sport,Venceslás Hoyos Palacios,plover@yahoo.com,(946) 962-4147,"2776,00",2016-11-13,null,2026-05-10T03:49:59.146Z,csv_provedores
C8748,P0221,Australia y Oceanía,Kiribati,Alimento infantil,Offline,Media,2009-01-02,20090102-3784,2009-01-04,"23163,00","263,19","164,36","6096355,21","3807117,47",2026-05-10T03:50:10.843Z,csv_general,C8748,Pepe Juanito Ad�n Anaya,1981-02-04,"Manotick, ON K4M 6S8",316 East Oak Meadow Court,(732) 906-0070,howler@mac.com,29/11/2019 13:47,B,2026-05-10T03:49:54.514Z,csv_clientes,P0221,Logique d'optimisation,Benigno Montalbán-Mascaró,aprakash@outlook.com,(241) 628-3119,"6196,00",2014-04-01,null,2026-05-10T03:49:59.146Z,csv_provedores
C8748,P0224,Australia y Oceanía,Kiribati,Alimento infantil,Offline,Media,2009-01-02,20090102-3056,2009-01-04,"89279,00","263,19","164,36","23497668,56","14674076,78",2026-05-10T03:50:10.843Z,csv_general,C8748,Pepe Juanito Ad�n Anaya,1981-02-04,"Manotick, ON K4M 6S8",316 East Oak Meadow Court,(732) 906-0070,howler@mac.com,29/11/2019 13:47,B,2026-05-10T03:49:54.514Z,csv_clientes,P0224,Usine 2cv,Régulo Plaza Jerez,salesgeek@mac.com,(876) 215-3650,"8023,00",2015-06-12,null,2026-05-10T03:49:59.146Z,csv_provedores
C8537,P0708,Australia y Oceanía,East Timor,Snacks,Online,Media,2009-01-02,20090102-1062,2009-02-05,"37249,00","157,31","100,46","5859639,45","3742058,38",2026-05-10T03:50:10.843Z,csv_general,C8537,Teresita Donaire Prieto,1988-06-14,45 South Street,London,(375) 276-9555,rnewman@yahoo.com,29/07/2006 2:48,C,2026-05-10T03:49:54.514Z,csv_clientes,P0708,Grans dissenys,Narcisa Camino Sevilla,luebke@aol.com,(561) 989-6966,"4107,00",2018-11-09,null,2026-05-10T03:49:59.146Z,csv_provedores


%md

# Construcción de la Capa Silver (`Build_Silver_Layer`)

## Descripción Funcional
Este componente realiza la depuración, normalización y estandarización de los datos provenientes de la capa Bronze. Se aplican transformaciones de tipado, limpieza de valores, eliminación de registros inválidos y enriquecimiento temporal, con el objetivo de mejorar la calidad y confiabilidad de la información de ventas.

## Objetivo dentro del Pipeline
Construir la capa Silver asegurando datos consistentes, limpios y estructurados, listos para análisis exploratorio, agregaciones y consumo por modelos analíticos o de Machine Learning.

## Entradas
- **Tabla origen:** `BRONZE_TABLE`
- **DataFrame:** `df_bronze_read`

## Procesamiento
- **Lectura de datos desde Bronze:**
  - `spark.table(BRONZE_TABLE)`

- **Estandarización y limpieza:**
  - Eliminación de espacios en columnas categóricas (`zona`, `canal_venta`, `prioridad`)
  - Conversión de tipos de datos:
    - `fecha_pedido`, `fecha_envio` → date
    - `precio_unitario`, `coste_unitario`, `importe_venta_total`, `importe_coste_total` → double / int
    - `ingestion_timestamp` → timestamp

- **Validaciones de calidad:**
  - Eliminación de registros con valores nulos en columnas críticas:
    - `id_cliente`, `id_proveedor`, `id_pedido`

- **Eliminación de duplicados:**
  - Basado en clave compuesta: (`id_pedido`)

- **Enriquecimiento de datos (Feature Engineering básico):**
  - `ANIO`: año del pedido
  - `MES`: mes del pedido
  - `DIA`: día de la semana del pedido

## Salidas
- **DataFrame:** `df_silver`
- Dataset limpio, tipado y enriquecido con variables temporales

## Capa del Pipeline
**Silver**

## Consideraciones Técnicas
- **Calidad de datos:**
  - Se eliminan registros incompletos (estrategia conservadora)
- **Tipado explícito:**
  - Mejora performance y evita errores en etapas posteriores
- **Deduplicación:**
  - Garantiza unicidad por pedido (`id_pedido`)
- **Feature Engineering inicial:**
  - Variables temporales útiles para análisis y modelos ML
- **Posible mejora:**
  - Implementar imputación en lugar de eliminación según el caso de uso
  - Validaciones más avanzadas (rangos, outliers, consistencia entre fechas)

## Ingenieria de datos
Esta fase transforma los datos "sucios" en datos listos para el análisis. Se eliminan espacios en blanco en categorías, se estandarizan las fechas de pedido y envío, y se realiza el casting de columnas monetarias de string a double. Se crea además la columna dias_entrega como una métrica operativa clave.



In [0]:
df_bronze_read = spark.table(BRONZE_TABLE)

df_silver = (
    df_bronze_read.select(

        F.col("ID_Cliente").cast("string").alias("id_cliente"),
        # Clientes
        F.col("CL_Nombre_completo").cast("string").alias("cl_nombre_completo"),
        F.col("CL_Grupo_de_clientes").cast("string").alias("cl_grupo_clientes"),
        # PROVEDORES
        F.col("id_provedor").cast("string").alias("id_proveedor"),
        F.col("PR_Proveedor").cast("string").alias("pr_proveedor"),
        F.col("PR_Contacto_comercial").cast("string").alias("pr_contacto_comercial"),
        # TABLA VENTAS
        F.col("Zona").cast("string").alias("zona"),
        F.col("País").cast("string").alias("pais"),
        F.col("Tipo_de_producto").cast("string").alias("tipo_producto"),
        F.col("Canal_de_venta").cast("string").alias("canal_venta"),
        F.col("Prioridad").cast("string").alias("prioridad"),        
        F.col("Fecha_pedido").cast("date").alias("fecha_pedido"),
        F.col("ID_Pedido").cast("string").alias("id_pedido"),
        F.col("Fecha_envío").cast("date").alias("fecha_envio"),
        F.regexp_replace(F.col("Unidades"),",",".").cast("double").cast("int").alias("Unidades"),
        F.regexp_replace(F.col("Precio_Unitario"),",",".").cast("double").alias("precio_unitario"),
        F.regexp_replace(F.col("Coste_unitario"),",",".").cast("double").cast("int").alias("coste_unitario"),
        F.regexp_replace(F.col("Importe_venta_total"),",",".").cast("double").cast("int").alias("importe_venta_total"),
        F.regexp_replace(F.col("Importe_Coste_total"),",",".").cast("double").cast("int").alias("importe_coste_total"),

        F.col("ingestion_timestamp").cast("timestamp").alias("ingestion_timestamp"),
        F.col("record_source").cast("string").alias("record_source"),
        F.col("PR_ingestion_timestamp").cast("timestamp").alias("pr_ingestion_timestamp")
    )

    # =========================
    # VALIDACIONES
    # =========================
    .filter(F.col("id_cliente").isNotNull())
    .filter(F.col("id_proveedor").isNotNull())
    .filter(F.col("id_pedido").isNotNull())

    # =========================
    # ELIMINAR DUPLICADOS
    # =========================
    .dropDuplicates(["id_pedido"])

    # =========================
    # CAMPOS DERIVADOS
    # =========================
    .withColumn("ANIO", F.year("fecha_pedido"))
    .withColumn("MES", F.month("fecha_pedido"))
    .withColumn("DIA", F.dayofweek("fecha_pedido"))
)

display(df_silver.limit(5))

id_cliente,cl_nombre_completo,cl_grupo_clientes,id_proveedor,pr_proveedor,pr_contacto_comercial,zona,pais,tipo_producto,canal_venta,prioridad,fecha_pedido,id_pedido,fecha_envio,Unidades,precio_unitario,coste_unitario,importe_venta_total,importe_coste_total,ingestion_timestamp,record_source,pr_ingestion_timestamp,ANIO,MES,DIA
C2421,Leandra Anna Malo Alba,A,P0001,So Factive,Lorenzo Cantón Galan,Europa,United Kingdom,Snacks,Offline,Crítica,2011-10-12,20111012-8455,2011-11-30,84173,159.45,101,13421056,8570898,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2011,10,4
C1908,Severo Granados Iglesia,E,P0002,Kontroller,Adelina Valls Canet,Europa,Malta,Cárnicos,Online,Alta,2011-01-26,20110126-3902,2011-01-28,19314,440.88,381,8515060,7360585,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2011,1,4
C7652,Lucho Andreu Amat,E,P0003,Finance Api,Eulalia del Galindo,Australia y Oceanía,Marshall Islands,Cereales,Online,Crítica,2011-11-09,20111109-1249,2011-11-21,2180,214.96,122,468605,266788,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2011,11,4
C2326,Mat�as Mauricio Castillo Barrera,E,P0004,Biomotivate,Tania Catalán Galván,África,Iran,Frutas,Offline,Baja,2012-08-21,20120821-8949,2012-10-02,20756,9.81,7,203529,150956,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2012,8,3
C5305,Mauricio Guijarro Castell�,D,P0005,Deltavita,Abraham Girón-Soler,Centroamérica y Caribe,Guatemala,Alimento infantil,Offline,Media,2013-09-30,20130930-4675,2013-11-12,73533,260.64,162,19165705,11968806,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2013,9,2


%md

# Persistencia de la Capa Silver (`Persist_Silver_Layer`)

## Descripción Funcional
Este componente realiza la persistencia del DataFrame transformado en la capa Silver, almacenándolo como una tabla Delta Lake. Se garantiza que los datos limpios, tipados y enriquecidos queden disponibles para consumo analítico y procesamiento posterior.

## Objetivo dentro del Pipeline
Materializar la capa Silver en almacenamiento persistente, asegurando disponibilidad de datos depurados y estructurados para la construcción de la capa Gold y modelos analíticos.

## Entradas
- **DataFrame origen:** `df_silver`
- **Tabla destino:** `SILVER_TABLE`
- **Formato de almacenamiento:** Delta Lake

## Procesamiento
- Eliminación previa de la tabla Silver (si existe):
  - `DROP TABLE IF EXISTS`
- Escritura del DataFrame:
  - Formato: `delta`
  - Modo: `overwrite`
  - Persistencia como tabla gestionada en el catálogo

## Salidas
- **Tabla Delta:** `SILVER_TABLE`
- Dataset limpio, consistente y listo para consumo

## Capa del Pipeline
**Silver**

## Consideraciones Técnicas
- **Modo overwrite:**
  - Reemplaza completamente la tabla en cada ejecución
- **Formato Delta:**
  - Soporte ACID
  - Versionado (Time Travel)
  - Optimización de consultas
- **Integridad de datos:**
  - Se asume que la calidad ya fue validada previamente
- **Escalabilidad:**
  - Optimizado para procesamiento distribuido en Spark



In [0]:
# Crear la capa silver 
spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
 
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)
 
print(f"Tabla Silver creada: {SILVER_TABLE}")

Tabla Silver creada: workspace.bigdata_proyecto.SILVER_Ventas_Proyecto


# Visualización de Datos Silver (`View_Silver_Data`)

## Descripción
Consulta y visualiza los datos de la tabla Silver ordenados por `zona` y `fecha_pedido`.

## Objetivo
Validar calidad, consistencia y estructura de los datos procesados.

## Entrada
- Tabla: `SILVER_TABLE`

## Procesamiento
- Lectura de tabla Delta
- Ordenamiento por `zona`, `fecha_pedido`
- Visualización con `display()`

## Salida
- Vista tabular para análisis exploratorio

## Capa
**Silver (Validación)**


In [0]:
display(spark.table(SILVER_TABLE).orderBy("zona", "fecha_pedido").limit(5))

id_cliente,cl_nombre_completo,cl_grupo_clientes,id_proveedor,pr_proveedor,pr_contacto_comercial,zona,pais,tipo_producto,canal_venta,prioridad,fecha_pedido,id_pedido,fecha_envio,Unidades,precio_unitario,coste_unitario,importe_venta_total,importe_coste_total,ingestion_timestamp,record_source,pr_ingestion_timestamp,ANIO,MES,DIA
C9679,Gloria Esteve Perez,D,P0334,Logiciel Smartworld,Salud Llorens-Carro,Asia,Taiwan,Bebida,Online,Media,2009-01-10,20090110-2838,2009-02-22,74518,48.92,32,3645491,2442363,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,7
C9679,Gloria Esteve Perez,D,P0337,Cq It Au Qatar,Angélica Purificación Vallejo Huerta,Asia,Taiwan,Bebida,Online,Media,2009-01-10,20090110-3029,2009-02-22,47158,48.92,32,2307014,1545626,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,7
C4617,Estela Cases Peral,E,P0234,Laboratoires Saatchi,Macarena Eufemia Llanos Herrera,Asia,Bhutan,Cosméticos,Offline,Crítica,2009-01-12,20090112-9665,2009-01-22,5718,450.75,271,2577406,1552398,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,2
C4617,Estela Cases Peral,E,P0237,Intercités,Marisela Girón Exposito,Asia,Bhutan,Cosméticos,Offline,Crítica,2009-01-12,20090112-3317,2009-01-22,74100,450.75,271,33400812,20117648,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,2
C8751,Am�rica Mor�n Pe�alver,A,P0251,Terrible herbe,Virginia Puga Prado,Asia,Brunei,Doméstico,Offline,Crítica,2009-01-16,20090116-9040,2009-02-06,47271,688.99,518,32569074,24491990,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,6


En el siguiente grafico de dispersión utilizando las variables unidades (eje X) e importe_venta_total (eje Y). El objetivo es validar la linealidad de los ingresos respecto al volumen y observar cómo se distribuyen las ventas según su Prioridad de Envío (Critical, High, Medium, Low), representada por la escala de colores. Este análisis permite identificar si la prioridad del pedido influye en el tamaño de la transacción o en el precio unitario promedio.

Conclusiones del Resultado (Outputs):
Correlación Lineal Perfecta: Se observa una relación lineal directa y positiva. Esto confirma que el modelo de precios es consistente: a mayor número de unidades, el importe crece de manera proporcional. No se detectan anomalías de precios (ventas con muchas unidades a precios extremadamente bajos o viceversa), lo cual indica que los datos en la capa Silver están bien depurados.

Distribución de Prioridades: Las categorías de prioridad (C, H, M, L) están uniformemente distribuidas a lo largo de toda la recta.

Reflexión: No existe un sesgo donde los pedidos "Críticos" sean exclusivamente de grandes volúmenes. Se atienden pedidos urgentes tanto de baja escala (cercanos al origen) como de gran escala (superiores a 10,000 unidades).

Densidad de Transacciones: Existe una alta concentración de puntos en el rango de 0 a 6,000 unidades. Las transacciones que superan las 8,000 unidades son menos frecuentes pero representan los picos de ingresos (superando los 4 millones en importe).

Identificación de Clústeres: Aunque la relación es lineal, se perciben ligeras variaciones en la pendiente (algunas líneas de puntos parecen más inclinadas que otras). Esto sugiere que el Tipo de Producto actúa como una variable moderadora; es decir, la pendiente representa el precio unitario del producto específico.

In [0]:
display(spark.table(SILVER_TABLE).orderBy("zona", "fecha_pedido"))

id_cliente,cl_nombre_completo,cl_grupo_clientes,id_proveedor,pr_proveedor,pr_contacto_comercial,zona,pais,tipo_producto,canal_venta,prioridad,fecha_pedido,id_pedido,fecha_envio,Unidades,precio_unitario,coste_unitario,importe_venta_total,importe_coste_total,ingestion_timestamp,record_source,pr_ingestion_timestamp,ANIO,MES,DIA
C9679,Gloria Esteve Perez,D,P0334,Logiciel Smartworld,Salud Llorens-Carro,Asia,Taiwan,Bebida,Online,Media,2009-01-10,20090110-2838,2009-02-22,74518,48.92,32,3645491,2442363,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,7
C9679,Gloria Esteve Perez,D,P0337,Cq It Au Qatar,Angélica Purificación Vallejo Huerta,Asia,Taiwan,Bebida,Online,Media,2009-01-10,20090110-3029,2009-02-22,47158,48.92,32,2307014,1545626,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,7
C4617,Estela Cases Peral,E,P0234,Laboratoires Saatchi,Macarena Eufemia Llanos Herrera,Asia,Bhutan,Cosméticos,Offline,Crítica,2009-01-12,20090112-9665,2009-01-22,5718,450.75,271,2577406,1552398,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,2
C4617,Estela Cases Peral,E,P0237,Intercités,Marisela Girón Exposito,Asia,Bhutan,Cosméticos,Offline,Crítica,2009-01-12,20090112-3317,2009-01-22,74100,450.75,271,33400812,20117648,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,2
C8751,Am�rica Mor�n Pe�alver,A,P0251,Terrible herbe,Virginia Puga Prado,Asia,Brunei,Doméstico,Offline,Crítica,2009-01-16,20090116-9040,2009-02-06,47271,688.99,518,32569074,24491990,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,6
C8751,Am�rica Mor�n Pe�alver,A,P0248,Anti-systèmes,Abilio Feliu Gelabert,Asia,Brunei,Doméstico,Offline,Crítica,2009-01-16,20090116-6057,2009-02-06,25665,688.99,518,17682835,13297517,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,6
C5338,Milagros Cadenas,C,P0580,Construtores de Goykay,Salud Larrea Álvaro,Asia,Myanmar,Doméstico,Offline,Media,2009-01-17,20090117-1331,2009-02-21,36023,688.99,518,24819356,18664191,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,7
C5338,Milagros Cadenas,C,P0583,Ciência sem limites,Nadia Maite Roldán Sacristán,Asia,Myanmar,Doméstico,Offline,Media,2009-01-17,20090117-6013,2009-02-21,29928,688.99,518,20619984,15506257,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,7
C9933,Sandalio Baeza Grande,E,P0200,Outsource It Development,Salomé Sales-Torres,Asia,Myanmar,Material de oficina,Online,Baja,2009-01-17,20090117-6228,2009-02-18,51672,671.4,541,34692452,27966630,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,7
C9933,Sandalio Baeza Grande,E,P0203,API Finance,Cruz Barroso-Flor,Asia,Myanmar,Material de oficina,Online,Baja,2009-01-17,20090117-8540,2009-02-18,62402,671.4,541,41896547,33774069,2026-05-10T03:50:10.843Z,csv_general,2026-05-10T03:49:59.146Z,2009,1,7


Databricks visualization. Run in Databricks to view.

## Descripción
Genera un conteo de registros por zona geográfica en la capa Silver.

## Objetivo
Verificar integridad y distribución de datos por zona.

## Entrada
- Tabla: `SILVER_TABLE`

## Procesamiento
- Agrupación por `zona`
- Conteo de registros
- Ordenamiento por `zona`

## Salida
- Conteo por zona (visualización)

## Capa
**Silver (Validación)**


In [0]:
# Validación de datos con un conteo por zona
print("Conteo por zona en Silver:")
display(
    spark.table(SILVER_TABLE)
    .groupBy("zona")
    .count()
    .orderBy("zona")
)


Conteo por zona en Silver:


zona,count
Asia,1944
Australia y Oceanía,1140
Centroamérica y Caribe,1224
Europa,2880
Norteamérica,300
África,4512


# Feature Engineering y Ventanas (`Build_Gold_Features`)

## Descripción
Genera variables analíticas usando ventanas deslizantes para preparar datos orientados a modelos de predicción de ventas.

## Objetivo
Construir features temporales, indicadores de rentabilidad y variable objetivo para ML.

## Entrada
- Tabla: `SILVER_TABLE`

## Procesamiento
- Creación de lags de importe de venta (lag 1, 2, 3 periodos)
- Cálculo de variación relativa de ventas (`var_venta_1m`, `var_venta_3m`)
- Promedios móviles de ventas (`ma_3`, `ma_7`)
- Volatilidad de ventas (`stddev_ventas_7m`)
- Variaciones porcentuales de unidades y costos
- Relación importe vs media móvil (`venta_vs_ma_7`)
- Margen de ganancia (`margen_ganancia`)
- Tiempo de entrega en días (`dias_entrega`)
- Variable objetivo (`target_alta_venta`): indica si la venta del siguiente periodo supera la actual

## Salida
- DataFrame: `df_gold` (features + target)

## Capa
**Gold (Feature Engineering / ML)**

## Procedimiento realizado
Se utiliza el concepto de Ventanas Deslizantes (Window Functions) particionadas por zona y tipo_producto. Se generan variables de retardo (Lags) para capturar la inercia de las ventas, medias móviles (ma_3, ma_7) para suavizar la volatilidad, y se calcula el Margen de Ganancia. Finalmente, se define target_alta_venta comparando la venta actual con la del periodo siguiente (lead), convirtiendo el problema en uno de Clasificación Binaria.
## Analisis
El conjunto de variables generado (22 columnas en total) permite capturar no solo el estado actual del negocio, sino su tendencia temporal. Variables como venta_vs_ma_7 son críticas para detectar si una venta actual es una anomalía o parte de un crecimiento sostenido.


In [0]:
df_silver_read = spark.table(SILVER_TABLE)

w_order  = Window.partitionBy("zona", "tipo_producto").orderBy("fecha_pedido")
w_3      = Window.partitionBy("zona", "tipo_producto").orderBy("fecha_pedido").rowsBetween(-2, 0)
w_7      = Window.partitionBy("zona", "tipo_producto").orderBy("fecha_pedido").rowsBetween(-6, 0)
w_target = Window.partitionBy("zona", "tipo_producto").orderBy("fecha_pedido")

df_gold = (
    df_silver_read
    # Lags de importe de venta
    .withColumn("lag_venta_1", F.lag("importe_venta_total", 1).over(w_order))
    .withColumn("lag_venta_2", F.lag("importe_venta_total", 2).over(w_order))
    .withColumn("lag_venta_3", F.lag("importe_venta_total", 3).over(w_order))
    # Lag de unidades y costo
    .withColumn("lag_unidades_1", F.lag("Unidades", 1).over(w_order))
    .withColumn("lag_coste_1",    F.lag("importe_coste_total", 1).over(w_order))
    # Valor futuro (target)
    .withColumn("next_venta", F.lead("importe_venta_total", 1).over(w_target))
    # Variaciones relativas de ventas
    .withColumn(
        "var_venta_1m",
        (F.col("importe_venta_total") - F.col("lag_venta_1")) / F.col("lag_venta_1")
    )
    .withColumn(
        "var_venta_3m",
        (F.col("importe_venta_total") - F.col("lag_venta_3")) / F.col("lag_venta_3")
    )
    # Medias móviles de ventas
    .withColumn("ma_3", F.avg("importe_venta_total").over(w_3))
    .withColumn("ma_7", F.avg("importe_venta_total").over(w_7))
    # Volatilidad de variaciones
    .withColumn("stddev_ventas_7m", F.stddev("var_venta_1m").over(w_7))
    # Variación de unidades
    .withColumn(
        "var_unidades_1m",
        (F.col("Unidades") - F.col("lag_unidades_1")) / F.col("lag_unidades_1")
    )
    # Variación de costo
    .withColumn(
        "var_coste_1m",
        (F.col("importe_coste_total") - F.col("lag_coste_1")) / F.col("lag_coste_1")
    )
    # Relación venta vs media móvil
    .withColumn("venta_vs_ma_7", F.col("importe_venta_total") / F.col("ma_7"))
    # Margen de ganancia
    .withColumn(
        "margen_ganancia",
        (F.col("importe_venta_total") - F.col("importe_coste_total")) / F.col("importe_venta_total")
    )
    # Tiempo de entrega en días
    .withColumn(
        "dias_entrega",
        F.datediff(F.col("fecha_envio"), F.col("fecha_pedido"))
    )
    # Variable objetivo: ¿la próxima venta supera la actual?
    .withColumn(
        "target_alta_venta",
        F.when(F.col("next_venta") > F.col("importe_venta_total"), 1).otherwise(0)
    )
)


# Selección y Limpieza Final (`Clean_Gold_Data`)

## Descripción
Filtra y selecciona variables relevantes, eliminando registros con valores nulos generados por las ventanas deslizantes.

## Objetivo
Garantizar un dataset completo y consistente para el entrenamiento del modelo ML.

## Entrada
- DataFrame: `df_gold`

## Procesamiento
- Selección de variables clave (features + target)
- Eliminación de nulos en variables críticas generadas por lags y ventanas

## Salida
- DataFrame: `df_gold_selected`

## Capa
**Gold (Preparación ML)**


In [0]:
df_gold_selected = (
    df_gold
    .select(
        "zona",
        "pais",
        "tipo_producto",
        "canal_venta",
        "prioridad",
        "fecha_pedido",
        "id_pedido",
        "ANIO",
        "MES",
        "DIA",
        "Unidades",
        "precio_unitario",
        "importe_venta_total",
        "importe_coste_total",
        "lag_venta_1",
        "lag_venta_2",
        "lag_venta_3",
        "var_venta_1m",
        "var_venta_3m",
        "ma_3",
        "ma_7",
        "stddev_ventas_7m",
        "var_unidades_1m",
        "var_coste_1m",
        "venta_vs_ma_7",
        "margen_ganancia",
        "dias_entrega",
        "target_alta_venta"
    )
    .filter(F.col("lag_venta_3").isNotNull())
    .filter(F.col("var_venta_1m").isNotNull())
    .filter(F.col("var_venta_3m").isNotNull())
    .filter(F.col("ma_7").isNotNull())
    .filter(F.col("stddev_ventas_7m").isNotNull())
    .filter(F.col("margen_ganancia").isNotNull())
    .filter(F.col("dias_entrega").isNotNull())
)


# Persistencia Capa Gold (`Persist_Gold_Layer`)

## Descripción
Guarda el dataset final (features + target) en Delta Lake como tabla Gold.

## Objetivo
Disponibilizar datos listos para analítica avanzada y modelos ML.

## Entrada
- DataFrame: `df_gold_selected`

## Procesamiento
- Drop tabla si existe
- Escritura en formato Delta (`overwrite`)

## Salida
- Tabla: `GOLD_TABLE`

## Capa
**Gold**


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {GOLD_TABLE}")
 
(
    df_gold_selected.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)
 
print(f"Tabla Gold creada: {GOLD_TABLE}")

Tabla Gold creada: workspace.bigdata_proyecto.GOLD_Ventas_Proyecto


# Visualización Capa Gold (`View_Gold_Data`)

## Descripción
Visualiza la evolución temporal de las métricas analíticas generadas (importe de venta, medias móviles, variaciones) desde la capa Gold.

## Objetivo
Analizar tendencias comerciales y validar el comportamiento de las variables generadas para ML.

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Lectura de datos desde Gold
- Ordenamiento por `zona`, `fecha_pedido`
- Visualización gráfica en Databricks

## Salida
- Vista tabular / gráfico de series temporales de ventas

## Capa
**Gold (Análisis / Validación)**


In [0]:
display(spark.table(GOLD_TABLE).orderBy("zona", "fecha_pedido").limit(5))

zona,pais,tipo_producto,canal_venta,prioridad,fecha_pedido,id_pedido,ANIO,MES,DIA,Unidades,precio_unitario,importe_venta_total,importe_coste_total,lag_venta_1,lag_venta_2,lag_venta_3,var_venta_1m,var_venta_3m,ma_3,ma_7,stddev_ventas_7m,var_unidades_1m,var_coste_1m,venta_vs_ma_7,margen_ganancia,dias_entrega,target_alta_venta
Asia,Myanmar,Doméstico,Offline,Media,2009-01-17,20090117-6013,2009,1,7,29928,688.99,20619984,15506257,24819356,32569074,17682835,-0.169197460240306,0.16610170258332446,2.6002804666666668E7,2.392281225E7,0.6045510991780086,-0.1691974571801349,-0.1691974755294778,0.8619381276965045,0.24799859204546423,35,1
Asia,Kazakhstan,Doméstico,Online,Media,2009-01-18,20090118-9499,2009,1,1,72989,688.99,50288426,37816968,37109494,20619984,24819356,0.35513639717103124,1.0261777138778299,3.6005968E7,3.05148615E7,0.5133844719996968,0.35513636954382577,0.35513636606493715,1.6479978452466513,0.247998575258649,24,0
Asia,Kazakhstan,Doméstico,Online,Media,2009-01-18,20090118-6695,2009,1,1,53861,688.99,37109494,27906393,20619984,24819356,32569074,0.7996858775448128,0.13940893744783778,2.7516278E7,2.65601486E7,0.5923179704644999,0.7996859128575248,0.7996859590293132,1.3971869871240101,0.24799855799704518,24,1
Asia,Maldives,Frutas,Offline,Alta,2009-02-10,20090210-7310,2009,2,3,93531,9.62,899696,667298,865374,468764,652578,0.03966146429173976,0.37867963676372784,744611.3333333334,721603.0,0.5810020164232775,0.03966074941920567,0.039660851112890715,1.2468019118545792,0.25830725044904057,3,0
Asia,Philippines,Cárnicos,Offline,Media,2009-02-28,20090228-3419,2009,2,7,79672,434.97,34654817,29956304,14336999,37974497,28325589,1.4171597556782978,0.22344559189925406,2.8988771E7,2.88229755E7,1.0203338931848671,1.4171596735535936,1.4171596258072179,1.2023330832030164,0.13558037256407962,29,0


# Análisis de Rentabilidad por Zona (`Analyze_Profitability`)

## Descripción
Calcula el importe de venta promedio, el margen de ganancia promedio y la variabilidad de ventas por zona geográfica y tipo de producto.

## Objetivo
Identificar las zonas y categorías de producto con mayor rentabilidad y menor volatilidad comercial.

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Agrupación por `zona` y `tipo_producto`
- Cálculo de:
  - `avg_venta`: importe de venta promedio
  - `avg_margen`: margen de ganancia promedio
  - `avg_volatilidad_ventas`: variabilidad promedio de las ventas
- Ordenamiento por importe de venta promedio descendente

## Salida
- Tabla comparativa de rentabilidad por zona y producto

## Capa
**Gold (Análisis)**


In [0]:
display(
    spark.sql(f"""
    SELECT 
        zona,
        tipo_producto,
        ROUND(AVG(importe_venta_total), 2)  AS avg_venta,
        ROUND(AVG(margen_ganancia), 4)       AS avg_margen,
        ROUND(AVG(stddev_ventas_7m), 6)      AS avg_volatilidad_ventas
    FROM {GOLD_TABLE}
    GROUP BY zona, tipo_producto
    ORDER BY avg_venta DESC
    limit(5)
    """)
)


zona,tipo_producto,avg_venta,avg_margen,avg_volatilidad_ventas
Asia,Doméstico,3.670836589E7,0.248,2.559019
Australia y Oceanía,Material de oficina,3.614426661E7,0.1939,2.247164
Norteamérica,Material de oficina,3.561778191E7,0.1939,6.435205
Europa,Doméstico,3.501564364E7,0.248,3.709633
Centroamérica y Caribe,Material de oficina,3.427661325E7,0.1939,2.572154


# Análisis de Tendencia de Ventas por Zona (`Analyze_Sales_Trend`)

## Descripción
Consulta la evolución temporal del importe de ventas de una zona seleccionada, junto con métricas derivadas.

## Objetivo
Analizar la tendencia, el comportamiento mensual y la relación con la media móvil para una zona específica.

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Filtro por `zona = 'Europe'` (ajustable según análisis requerido)
- Selección de variables: `importe_venta_total`, `ma_7`, `var_venta_1m`, `margen_ganancia`
- Ordenamiento por `fecha_pedido`

## Salida
- Serie temporal de ventas para la zona seleccionada

## Capa
**Gold (Análisis)**


In [0]:
display(
    spark.sql(f"""
    SELECT 
        zona,
        fecha_pedido,
        tipo_producto,
        importe_venta_total,
        ma_7,
        var_venta_1m,
        margen_ganancia
    FROM {GOLD_TABLE}
    WHERE zona = 'Europa'
    ORDER BY fecha_pedido
    limit(5)
    """))


zona,fecha_pedido,tipo_producto,importe_venta_total,ma_7,var_venta_1m,margen_ganancia
Europa,2009-01-07,Frutas,433856,461559.0,-0.11004262546615193,0.2583069036731081
Europa,2009-01-08,Frutas,835737,536394.6,0.9263004314795693,0.2583061417646939
Europa,2009-01-08,Frutas,847357,588221.6666666666,0.013903895603521203,0.25830671133890437
Europa,2009-01-09,Frutas,243472,538971.7142857143,-0.7126689223078349,0.2583089636590655
Europa,2009-01-09,Frutas,952909,614969.1428571428,2.9138340343037394,0.2583069317217069


# Balance de Clases (`Check_Class_Balance`)

## Descripción
Calcula la distribución de la variable objetivo `target_alta_venta` por zona geográfica.

## Objetivo
Evaluar el desbalance de clases antes del entrenamiento del modelo ML, permitiendo tomar decisiones sobre técnicas de balanceo (oversampling, undersampling, pesos de clase).

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Agrupación por `zona` y `target_alta_venta`
- Conteo de registros por clase
- Ordenamiento por zona y clase

## Salida
- Distribución de clases por zona geográfica

## Capa
**Gold (Validación ML)**


In [0]:
# Balance de las clases por zona
display(
    spark.sql(f"""
    SELECT
        zona,
        target_alta_venta,
        COUNT(*) AS total_registros
    FROM {GOLD_TABLE}
    GROUP BY zona, target_alta_venta
    ORDER BY zona, target_alta_venta
    """)
)


zona,target_alta_venta,total_registros
Asia,0,991
Asia,1,917
Australia y Oceanía,0,575
Australia y Oceanía,1,529
Centroamérica y Caribe,0,608
Centroamérica y Caribe,1,580
Europa,0,1442
Europa,1,1402
Norteamérica,0,147
Norteamérica,1,123


# Preparación Dataset ML (`Prepare_ML_Dataset`)

## Descripción
Construye el dataset final para entrenamiento del modelo de clasificación.

## Objetivo
Definir variables predictoras (features) y variable objetivo (`target_alta_venta`) orientadas a predecir si el importe de venta del siguiente periodo superará al actual.

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Selección de columnas:
  - Target: `target_alta_venta`
  - Features: variables numéricas, temporales y de comportamiento comercial
- Ordenamiento por `zona`, `fecha_pedido`

## Salida
- DataFrame: `df_ml`

## Capa
**Gold (ML)**


In [0]:
df_ml = spark.table(GOLD_TABLE)

feature_cols = [
    "ANIO",
    "MES",
    "DIA",
    "Unidades",
    "precio_unitario",
    "importe_coste_total",
    "lag_venta_1",
    "lag_venta_2",
    "lag_venta_3",
    "var_venta_1m",
    "var_venta_3m",
    "ma_3",
    "ma_7",
    "stddev_ventas_7m",
    "var_unidades_1m",
    "var_coste_1m",
    "venta_vs_ma_7",
    "margen_ganancia",
    "dias_entrega"
]

df_ml = df_ml.select("zona", "tipo_producto", "fecha_pedido", "target_alta_venta", *feature_cols)
display(df_ml.orderBy("zona", "fecha_pedido").limit(5))


zona,tipo_producto,fecha_pedido,target_alta_venta,ANIO,MES,DIA,Unidades,precio_unitario,importe_coste_total,lag_venta_1,lag_venta_2,lag_venta_3,var_venta_1m,var_venta_3m,ma_3,ma_7,stddev_ventas_7m,var_unidades_1m,var_coste_1m,venta_vs_ma_7,margen_ganancia,dias_entrega
Asia,Doméstico,2009-01-17,1,2009,1,7,29928,688.99,15506257,24819356,32569074,17682835,-0.169197460240306,0.16610170258332446,2.6002804666666668E7,2.392281225E7,0.6045510991780086,-0.1691974571801349,-0.1691974755294778,0.8619381276965045,0.24799859204546423,35
Asia,Doméstico,2009-01-18,0,2009,1,1,72989,688.99,37816968,37109494,20619984,24819356,0.35513639717103124,1.0261777138778299,3.6005968E7,3.05148615E7,0.5133844719996968,0.35513636954382577,0.35513636606493715,1.6479978452466513,0.247998575258649,24
Asia,Doméstico,2009-01-18,1,2009,1,1,53861,688.99,27906393,20619984,24819356,32569074,0.7996858775448128,0.13940893744783778,2.7516278E7,2.65601486E7,0.5923179704644999,0.7996859128575248,0.7996859590293132,1.3971869871240101,0.24799855799704518,24
Asia,Frutas,2009-02-10,0,2009,2,3,93531,9.62,667298,865374,468764,652578,0.03966146429173976,0.37867963676372784,744611.3333333334,721603.0,0.5810020164232775,0.03966074941920567,0.039660851112890715,1.2468019118545792,0.25830725044904057,3
Asia,Cárnicos,2009-02-28,0,2009,2,7,79672,434.97,29956304,14336999,37974497,28325589,1.4171597556782978,0.22344559189925406,2.8988771E7,2.88229755E7,1.0203338931848671,1.4171596735535936,1.4171596258072179,1.2023330832030164,0.13558037256407962,29


# Definición Rango Temporal (`Define_Date_Range`)

## Descripción
Obtiene las fechas mínima y máxima del dataset de ventas.

## Objetivo
Establecer los límites temporales para la división cronológica train/test, garantizando la integridad de la serie de tiempo de ventas.

## Entrada
- DataFrame: `df_ml`

## Procesamiento
- Cálculo de `min(fecha_pedido)` y `max(fecha_pedido)`
- Extracción de valores a variables locales

## Salida
- `min_date`, `max_date`

## Capa
**Gold (Preparación ML)**


In [0]:
date_bounds = df_ml.agg(
    F.min("fecha_pedido").alias("min_date"),
    F.max("fecha_pedido").alias("max_date")
).collect()[0]

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]

print("Fecha mínima:", min_date)
print("Fecha máxima:", max_date)


Fecha mínima: 2009-01-07
Fecha máxima: 2026-11-07


# División Cronológica Train/Test (`Time_Series_Split`)

## Descripción
Divide el dataset en conjuntos de entrenamiento y prueba respetando el orden cronológico de las ventas.

## Objetivo
Mantener la integridad temporal de la serie de ventas para evitar fuga de información (data leakage) en el entrenamiento del modelo.

## Entrada
- DataFrame: `df_ml`

## Procesamiento
- Obtención de fechas únicas ordenadas de `fecha_pedido`
- Cálculo del corte en el percentil 80% (80% train / 20% test)
- Definición de `split_date`

## Salida
- Fecha de corte: `split_date`

## Capa
**Gold (ML - Split Temporal)**


In [0]:
# Percentil temporal aproximado usando orden cronológico de fecha_pedido
distinct_dates = (
    df_ml.select("fecha_pedido")
    .distinct()
    .orderBy("fecha_pedido")
    .toPandas()["fecha_pedido"]
    .tolist()
)

split_idx  = int(len(distinct_dates) * 0.8)
split_date = distinct_dates[split_idx]

print("Fecha de corte (split_date):", split_date)


Fecha de corte (split_date): 2023-05-09


# Generación de Sets Train/Test (`Create_Train_Test_Sets`)

## Descripción
Separa el dataset en conjuntos de entrenamiento y prueba basándose en la fecha de corte cronológica.

## Objetivo
Preparar los datos para entrenamiento y evaluación del modelo de predicción de ventas, respetando el orden temporal.

## Entrada
- DataFrame: `df_ml`
- Fecha de corte: `split_date`

## Procesamiento
- `train_df`: registros con `fecha_pedido` anterior al split (80%)
- `test_df`: registros con `fecha_pedido` posterior o igual al split (20%)
- Conteo de registros por conjunto

## Salida
- DataFrames: `train_df`, `test_df`

## Capa
**Gold (ML)**


In [0]:
train_df = df_ml.filter(F.col("fecha_pedido") < F.lit(split_date))
test_df  = df_ml.filter(F.col("fecha_pedido") >= F.lit(split_date))

print("Registros de entrenamiento:", train_df.count())
print("Registros de prueba:", test_df.count())


Registros de entrenamiento: 9372
Registros de prueba: 2418


# Pipeline de Modelado (`Build_ML_Pipeline`)

## Descripción
Construye un pipeline de ML con indexación de variables categóricas, ensamblado de features y Regresión Logística para clasificación de ventas.

## Objetivo
Predecir si el importe de venta del siguiente periodo superará al actual (`target_alta_venta = 1`), usando variables históricas de ventas, costos, márgenes y comportamiento temporal.

## Entrada
- DataFrames: `train_df`, `test_df`
- Features: `feature_cols`

## Procesamiento
- Codificación de variable categórica (`zona`)
- Ensamble de todas las variables en vector (`features`)
- Aplicación de modelo:
  - Regresión Logística (`maxIter=100`)

## Salida
- Pipeline: `lr_pipeline`

## Capa
**Gold (ML - Modelado)**


In [0]:
zona_indexer = StringIndexer(
    inputCol="zona",
    outputCol="zona_index",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=["zona_index"] + feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="target_alta_venta",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=100,
    regParam=0.0
)

lr_pipeline = Pipeline(stages=[zona_indexer, assembler, lr])


# Entrenamiento y Predicción (`Train_Predict_Model`)

## Descripción
Entrena el modelo de Regresión Logística y genera predicciones sobre el conjunto de prueba.

## Objetivo
Evaluar la capacidad del modelo para predecir si el importe de venta del siguiente periodo será superior al actual.

## Entrada
- `train_df`, `test_df`
- Pipeline: `lr_pipeline`

## Procesamiento
- Entrenamiento del modelo (`fit`) con datos históricos de ventas
- Generación de predicciones (`transform`) sobre el conjunto de prueba

## Salida
- DataFrame: `lr_predictions` (incluye `prediction` y `probability`)

## Capa
**Gold (ML - Predicción)**


In [0]:
lr_model       = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

display(
    lr_predictions.select(
        "zona",
        "fecha_pedido",
        "target_alta_venta",
        "prediction",
        "probability"
    ).orderBy("zona", "fecha_pedido").limit(5)
)


zona,fecha_pedido,target_alta_venta,prediction,probability
Asia,2023-05-10,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8418675687291403"",""0.15813243127085974""]}"
Asia,2023-05-10,1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.39790109324431266"",""0.6020989067556873""]}"
Asia,2023-05-12,1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5714398037600793"",""0.4285601962399207""]}"
Asia,2023-05-12,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9016659679889897"",""0.09833403201101032""]}"
Asia,2023-05-23,1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0800065917004657"",""0.9199934082995342""]}"


# Entrenamiento y Predicción — Verificación (`Train_Predict_Model`)

## Descripción
Reejecutar el entrenamiento y predicción del modelo de Regresión Logística para confirmación de resultados.

## Objetivo
Predecir si el importe de venta del siguiente periodo supera al actual (`target_alta_venta`).

## Entrada
- `train_df`, `test_df`
- Pipeline: `lr_pipeline`

## Procesamiento
- Entrenamiento del modelo (`fit`)
- Generación de predicciones (`transform`)

## Salida
- DataFrame: `lr_predictions` (incluye `prediction`, `probability`)

## Capa
**Gold (ML)**


In [0]:
lr_model       = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

display(
    lr_predictions.select(
        "zona",
        "fecha_pedido",
        "target_alta_venta",
        "prediction",
        "probability"
    ).orderBy("zona", "fecha_pedido").limit(5)
)


zona,fecha_pedido,target_alta_venta,prediction,probability
Asia,2023-05-10,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8418675687291403"",""0.15813243127085974""]}"
Asia,2023-05-10,1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.39790109324431266"",""0.6020989067556873""]}"
Asia,2023-05-12,1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5714398037600793"",""0.4285601962399207""]}"
Asia,2023-05-12,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9016659679889897"",""0.09833403201101032""]}"
Asia,2023-05-23,1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0800065917004657"",""0.9199934082995342""]}"


# Evaluación del Modelo (`Evaluate_Model`)

## Descripción
Define los evaluadores de métricas para medir el desempeño del modelo de clasificación de ventas.

## Objetivo
Cuantificar la capacidad predictiva del modelo sobre datos históricos de ventas.

## Entrada
- DataFrame: `lr_predictions`

## Procesamiento
- Métricas calculadas:
  - AUC (ROC): discriminación entre alta y baja venta
  - Accuracy: porcentaje de predicciones correctas
  - F1 Score: balance entre precision y recall
  - Precision ponderada
  - Recall ponderado

## Salida
- Evaluadores y métricas (`auc_lr`, evaluadores multi-clase)

## Capa
**Gold (ML - Evaluación)**


In [0]:
binary_eval = BinaryClassificationEvaluator(
    labelCol="target_alta_venta",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_lr = binary_eval.evaluate(lr_predictions)

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="target_alta_venta",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="target_alta_venta",
    predictionCol="prediction",
    metricName="f1"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol="target_alta_venta",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="target_alta_venta",
    predictionCol="prediction",
    metricName="weightedRecall"
)


# Resultados del Modelo (`Model_Metrics_Results`)

## Descripción
Calcula y muestra las métricas finales del modelo de Regresión Logística sobre el conjunto de prueba de ventas.

## Objetivo
Cuantificar el desempeño del modelo de clasificación de alta/baja venta.

## Entrada
- DataFrame: `lr_predictions`

## Procesamiento
- Evaluación de métricas:
  - AUC ROC
  - Accuracy
  - F1-score
  - Precision ponderada
  - Recall ponderado

## Salida
- Métricas impresas en consola

## Capa
**Gold (ML - Evaluación)**


In [0]:
accuracy_lr  = accuracy_eval.evaluate(lr_predictions)
f1_lr        = f1_eval.evaluate(lr_predictions)
precision_lr = precision_eval.evaluate(lr_predictions)
recall_lr    = recall_eval.evaluate(lr_predictions)

print(f"AUC ROC (Logistic Regression): {auc_lr:.4f}")
print(f"Accuracy (Logistic Regression): {accuracy_lr:.4f}")
print(f"F1-score (Logistic Regression): {f1_lr:.4f}")
print(f"Weighted Precision (Logistic Regression): {precision_lr:.4f}")
print(f"Weighted Recall (Logistic Regression): {recall_lr:.4f}")


AUC ROC (Logistic Regression): 0.8241
Accuracy (Logistic Regression): 0.7448
F1-score (Logistic Regression): 0.7449
Weighted Precision (Logistic Regression): 0.7451
Weighted Recall (Logistic Regression): 0.7448


# Modelo Random Forest (`Build_RF_Model`)

## Descripción
Implementa un modelo de clasificación basado en Random Forest para predecir el comportamiento de ventas.

## Objetivo
Capturar relaciones no lineales entre las variables de ventas, costos, márgenes y comportamiento temporal para mejorar la capacidad predictiva frente a la Regresión Logística.

## Entrada
- `train_df`, `test_df`
- Features: `feature_cols`

## Procesamiento
- Ensamble de variables en vector `features`
- Entrenamiento Random Forest (`numTrees=100`, `maxDepth=6`, `seed=42`)
- Generación de predicciones sobre el conjunto de prueba

## Salida
- DataFrame: `rf_predictions`

## Capa
**Gold (ML - Modelado)**


In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="target_alta_venta",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=6,
    seed=42
)

rf_pipeline = Pipeline(stages=[zona_indexer, assembler, rf])
rf_model    = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)


# Evaluación Random Forest (`Evaluate_RF_Model`)

## Descripción
Calcula las métricas de desempeño del modelo Random Forest sobre el conjunto de prueba de ventas.

## Objetivo
Comparar el rendimiento del Random Forest frente a la Regresión Logística en la predicción de alta/baja venta.

## Entrada
- DataFrame: `rf_predictions`

## Procesamiento
- Evaluación de métricas:
  - AUC ROC
  - Accuracy
  - F1-score
  - Precision ponderada
  - Recall ponderado

## Salida
- Métricas impresas en consola

## Capa
**Gold (ML - Evaluación)**


In [0]:
auc_rf       = binary_eval.evaluate(rf_predictions)
accuracy_rf  = accuracy_eval.evaluate(rf_predictions)
f1_rf        = f1_eval.evaluate(rf_predictions)
precision_rf = precision_eval.evaluate(rf_predictions)
recall_rf    = recall_eval.evaluate(rf_predictions)

print(f"AUC ROC (Random Forest): {auc_rf:.4f}")
print(f"Accuracy (Random Forest): {accuracy_rf:.4f}")
print(f"F1-score (Random Forest): {f1_rf:.4f}")
print(f"Weighted Precision (Random Forest): {precision_rf:.4f}")
print(f"Weighted Recall (Random Forest): {recall_rf:.4f}")


AUC ROC (Random Forest): 0.8204
Accuracy (Random Forest): 0.7436
F1-score (Random Forest): 0.7436
Weighted Precision (Random Forest): 0.7436
Weighted Recall (Random Forest): 0.7436


# Comparación de Modelos (`Compare_Models`)

## Descripción
Consolida y compara las métricas de los modelos evaluados sobre el dataset de ventas.

## Objetivo
Identificar el modelo con mejor desempeño predictivo para la clasificación de alta/baja venta en el contexto comercial.

## Entrada
- Métricas: Logistic Regression, Random Forest

## Procesamiento
- Creación de DataFrame comparativo con métricas por modelo
- Ordenamiento y visualización

## Salida
- DataFrame: `metrics_df`

## Capa
**Gold (ML - Evaluación Comparativa)**


A continuacion se muestra el desempeño los resultados del modelo de clasificación para predecir el Target de Alta Venta. Los resultados indican que se cuenta con un modelo con una capacidad predictiva sólida, pero con áreas de mejora en la precisión de las clases positivas.

## Evaluación del Modelo de Machine Learning: Resultados de Clasificación
Explicación del Procedimiento:
Tras entrenar el modelo en la capa Gold utilizando un split temporal, se procedió a evaluar su capacidad de generalización sobre el conjunto de prueba (test). Se generó una Matriz de Confusión para comparar las predicciones del modelo frente a los valores reales y un Reporte de Clasificación que detalla métricas clave como Precision, Recall y F1-Score para ambas clases (0: Venta Normal, 1: Alta Venta).

Análisis de la Matriz de Confusión (Outputs):
Verdaderos Negativos (1177): El modelo es excelente identificando las ventas que no serán excepcionalmente altas.

Verdaderos Positivos (719): El modelo logra capturar una cantidad importante de casos de éxito comercial.

Falsos Positivos (218): En 218 casos, el modelo predijo una "Alta Venta" que no ocurrió. Esto podría llevar a un exceso de inventario o sobreestimación de ingresos.

Falsos Negativos (304): Estos son "costos de oportunidad". El modelo no detectó 304 ventas altas, lo que significa que la empresa no se preparó para esa demanda extra.

Interpretación de Métricas:
Accuracy (79%): El modelo acierta en casi 8 de cada 10 predicciones. Para un modelo de Big Data con alta variabilidad, es un resultado inicial muy prometedor.

Precision - Clase 1 (77%): Cuando el modelo dice que habrá una "Alta Venta", tiene un 77% de probabilidad de estar en lo cierto.

Recall - Clase 1 (70%): El modelo es capaz de detectar el 70% del total de las altas ventas reales. Este es el punto principal a mejorar; idealmente, querríamos capturar más casos positivos.

F1-Score (0.73): El balance entre precisión y sensibilidad es saludable, lo que indica que el modelo no está sesgado excesivamente hacia ninguna de las dos clases.

Conclusiones y Siguientes Pasos:
Reflexión Técnica: La diferencia entre el acierto de la clase 0 (80%) y la clase 1 (70%) sugiere que los patrones de "Venta Normal" son más estables y fáciles de aprender que los de "Alta Venta", los cuales suelen estar influenciados por factores externos no presentes en el dataset (promociones, factores políticos, etc.).

In [0]:
metrics_data = [
    ("LogisticRegression", auc_lr, accuracy_lr, f1_lr, precision_lr, recall_lr),
    ("RandomForest",       auc_rf, accuracy_rf, f1_rf, precision_rf, recall_rf),
]

metrics_df = spark.createDataFrame(
    metrics_data,
    ["modelo", "auc_roc", "accuracy", "f1_score", "precision_ponderada", "recall_ponderado"]
)

display(metrics_df)


modelo,auc_roc,accuracy,f1_score,precision_ponderada,recall_ponderado
LogisticRegression,0.824082341405696,0.7448304383788255,0.7448823450301141,0.7451485043276175,0.7448304383788255
RandomForest,0.8204235526438303,0.7435897435897436,0.7435897435897436,0.7435897435897436,0.7435897435897436


# Reflexión General
El modelo Random Forest aplicado al dataset de ventas históricas globales busca capturar patrones comerciales no lineales relacionados con zonas geográficas, tipos de producto, canales de venta, márgenes y comportamiento temporal. La calidad predictiva dependerá de la riqueza de las features generadas y de la consistencia del dato histórico procesado en las capas Bronze y Silver.

# Conclusiones del Modelo Random Forest

## 1. Capacidad discriminativa
Un **AUC ROC cercano o superior a 0.6** indica que el modelo comienza a capturar señales comerciales relevantes. Valores por debajo de 0.5 sugieren que las features seleccionadas no son suficientemente informativas para el horizonte de predicción definido.

## 2. Precisión global
**Accuracy esperado: entre 50% y 70%** dependiendo del balanceo de clases y la riqueza de las variables predictoras. Un accuracy inferior al 50% indica que el modelo no supera un clasificador trivial.

## 3. Desempeño por métricas
- **F1-score:** Mide el balance entre precisión y recall. Valores bajos pueden indicar desbalance de clases o features poco informativas.
- **Precision:** Calidad de las predicciones positivas (ventas altas identificadas correctamente).
- **Recall:** Capacidad del modelo para detectar todos los periodos de alta venta real.

## 4. Posibles causas de bajo desempeño
Desde una perspectiva de Data Science aplicada a ventas:
- Granularidad del dataset (nivel pedido vs. nivel agregado mensual)
- Features poco informativas o con alta correlación entre sí
- Desbalance de clases en `target_alta_venta`
- Horizonte de predicción a corto plazo con alta variabilidad
- Estacionalidad no capturada adecuadamente en los lags

## 5. Implicaciones y recomendaciones
- Enriquecer features con variables de estacionalidad (trimestre, festivos, campañas)
- Agregar datos externos (macro-económicos, tasas de cambio por región)
- Explorar modelos de series de tiempo (Prophet, LSTM) para capturar patrones temporales complejos
- Evaluar la predicción a nivel mensual o trimestral en lugar de por pedido
- Aplicar técnicas de balanceo de clases (SMOTE, pesos de clase) si hay desbalance significativo
